In [2]:

import pandas as pd
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

data_path = PROJECT_DIR / "data" / "processed" / "ted_talks_cleaned.csv"

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nAvailable columns:")
print(df.columns.tolist())


Dataset loaded successfully!
Dataset shape: (4005, 24)

Available columns:
['talk_id', 'title', 'speaker_1', 'all_speakers', 'occupations', 'about_speakers', 'views', 'recorded_date', 'published_date', 'event', 'native_lang', 'available_lang', 'comments', 'duration', 'topics', 'related_talks', 'url', 'description', 'transcript', 'title_clean', 'topics_clean', 'description_clean', 'transcript_clean', 'combined_text']


In [5]:

print("Title column available:", "title" in df.columns)
print("Combined text column available:", "combined_text" in df.columns)

usable_records = df[
    df["title"].notna()
    & df["combined_text"].notna()
    & df["title"].astype(str).str.strip().ne("")
    & df["combined_text"].astype(str).str.strip().ne("")
]

print("Total records:", len(df))
print("Usable records:", len(usable_records))


Title column available: True
Combined text column available: True
Total records: 4005
Usable records: 4005


In [6]:

# Import TF-IDF vectorizer to convert text into numerical features
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the vectorizer and keep at most 10,000 vocabulary terms
vectorizer = TfidfVectorizer(max_features=10000)

# Learn the vocabulary and convert each TED Talk's combined text into numbers
tfidf_matrix = vectorizer.fit_transform(usable_records["combined_text"])

# Display the results of text vectorization
print("TF-IDF vectorization completed!")

# Show the number of talks (rows) and vocabulary terms (columns)
print("Matrix dimensions:", tfidf_matrix.shape)

# Show how many unique vocabulary terms were learned
print("Vocabulary size:", len(vectorizer.vocabulary_))


TF-IDF vectorization completed!
Matrix dimensions: (4005, 10000)
Vocabulary size: 10000


In [7]:

# Get the vocabulary terms learned by the TF-IDF vectorizer
vocabulary = vectorizer.get_feature_names_out()

# Display the first 20 terms in the vocabulary
print("Sample vocabulary terms:")
print(vocabulary[:20])



Sample vocabulary terms:
['00' '000' '10' '100' '101' '10th' '11' '110' '11th' '12' '120' '125'
 '12th' '13' '130' '13th' '14' '140' '15' '150']


In [18]:
# Calculate cosine similarity for the first TED Talk
from sklearn.metrics.pairwise import cosine_similarity

# Select the first TED Talk
selected_talk = tfidf_matrix[0]

# Compare it with all TED Talks
similarity_scores = cosine_similarity(selected_talk, tfidf_matrix)[0]

print("Cosine similarity calculated successfully.")
# Display similarity scores for the first TED Talk
print(similarity_scores)

Cosine similarity calculated successfully.
[1.         0.64916762 0.68708071 ... 0.51612732 0.33653202 0.35895341]


In [20]:
# Find talks with the highest similarity scores
similar_talk_indices = similarity_scores.argsort()[::-1]

# Display the five most similar talks, excluding itself
count = 0

for index in similar_talk_indices:
    if index != 1:
        print("Title:", usable_records.iloc[index]["title"])
        print("Similarity score:", similarity_scores[index])
        print()
        count += 1

        if count == 5:
            break

Title: Averting the climate crisis
Similarity score: 0.9999999999999998

Title: Design and discovery
Similarity score: 0.7501634126112364

Title: New thinking on the climate crisis
Similarity score: 0.7333921414628675

Title: 5 predictions, from 1984
Similarity score: 0.7329127002446434

Title: The ghastly tragedy of the suburbs
Similarity score: 0.7299020573546244



In [26]:
# Check the selected TED Talk's title
print(usable_records.iloc[0]["title"])

Averting the climate crisis


In [27]:
# Compare the selected talk with the first talk
print("Selected talk:", usable_records.iloc[1]["title"])
print("First talk:", usable_records.iloc[0]["title"])
print("Similarity score:", similarity_scores[0])

Selected talk: The best stats you've ever seen
First talk: Averting the climate crisis
Similarity score: 0.9999999999999998


In [28]:
# Generate the top 5 recommendations for 3 TED Talks
from sklearn.metrics.pairwise import cosine_similarity

selected_indices = [0, 1, 2]

for talk_index in selected_indices:
    # Calculate similarity scores
    scores = cosine_similarity(
        tfidf_matrix[talk_index], tfidf_matrix
    )[0]

    # Sort talks by similarity
    similar_indices = scores.argsort()[::-1]

    print("\nSelected TED Talk:", usable_records.iloc[talk_index]["title"])
    print("Top 5 recommendations:")

    count = 0
    for index in similar_indices:
        if index != talk_index:
            print(
                usable_records.iloc[index]["title"],
                "| Similarity score:", round(scores[index], 4)
            )
            count += 1

            if count == 5:
                break


Selected TED Talk: Averting the climate crisis
Top 5 recommendations:
Design and discovery | Similarity score: 0.7502
New thinking on the climate crisis | Similarity score: 0.7334
5 predictions, from 1984 | Similarity score: 0.7329
The ghastly tragedy of the suburbs | Similarity score: 0.7299
My wish: Manufactured landscapes and green education | Similarity score: 0.7276

Selected TED Talk: The best stats you've ever seen
Top 5 recommendations:
Let my dataset change your mindset | Similarity score: 0.8581
New insights on poverty | Similarity score: 0.8154
How not to be ignorant about the world | Similarity score: 0.7293
The good news of the decade? We're winning the war against child mortality | Similarity score: 0.7264
Asia's rise -- how and when | Similarity score: 0.7238

Selected TED Talk: Simplicity sells
Top 5 recommendations:
Design and discovery | Similarity score: 0.7515
5 predictions, from 1984 | Similarity score: 0.7367
Smart failure for a fast-changing world | Similarity s

### Manual Review of Recommendations

1. For "Averting the climate crisis", "New thinking on the climate crisis" appears relevant because both titles relate to the climate crisis.

2. For "The best stats you've ever seen", "Let my dataset change your mindset" appears potentially relevant because both may involve data and statistics.

3. For "Simplicity sells", "Design and discovery" appears potentially relevant because both may discuss design or innovation.

### Limitations of TF-IDF Recommendations

1. Talks with similar words may be recommended even when their actual meanings differ.

2. Talks about similar topics may receive lower similarity scores if they use different vocabulary.

3. A high similarity score does not guarantee that a user will like or find a recommended talk useful.